Model LSTM

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tabulate import tabulate
import tensorflow as tf
import random
import os

seeds = [1, 444, 31, 45, 5]
precisions = []
somme_hausse = []

# --- 1. Chargement des données ---
df = pd.read_csv("Données_Diff.csv", parse_dates=True, index_col=0)
df = df.sort_index()

# --- Nettoyage AVANT normalisation ---
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=df.columns)  # supprime toutes les lignes avec NaN/Inf

# --- 2. Préparation des features et target ---
features = df.drop(columns=["result"])
target = df["result"]
y_binary = np.where(target == 1, 1, 0)

# --- 3. Normalisation ---
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(features)

# --- 4. Création des séquences ---
TIME_STEPS = 20
FUTURE_STEPS = 10

def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_binary, TIME_STEPS)
X_train, y_train = X_seq[:-FUTURE_STEPS], y_seq[:-FUTURE_STEPS]

# --- 5. Entraînement et évaluation du modèle pour chaque seed ---
for seed in seeds:
    # Fixer les seeds pour la reproductibilité
    np.random.seed(seed)
    tf.random.set_seed(seed)
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

    # --- 5. Modèle LSTM ---
    model = Sequential()
    model.add(LSTM(64, input_shape=(TIME_STEPS, X_train.shape[2])))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])

    # --- 6. Entraînement ---
    model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

    # --- 10. Prédiction sur les 10 derniers jours réels du CSV ---
    X_test_last10 = X_seq[-FUTURE_STEPS:]
    y_test_last10 = y_seq[-FUTURE_STEPS:]

    probas_last10 = model.predict(X_test_last10).flatten()
    preds_last10 = (probas_last10 > 0.5).astype(int)

    accuracy = np.mean(preds_last10 == y_test_last10)
    precisions.append(accuracy)

    # --- Somme de Diff_jour pour les prédictions "Hausse" ---
    probas_last10 = model.predict(X_test_last10).flatten()
    preds_last10 = (probas_last10 > 0.5).astype(int)

    accuracy = np.mean(preds_last10 == y_test_last10)
    precisions.append(accuracy)

    # On récupère les index des 10 derniers jours dans le DataFrame d'origine
    idx_last10 = df.index[-FUTURE_STEPS:]
    diff_jour_last10 = df.loc[idx_last10, "Diff_jour"].values

    somme = diff_jour_last10[preds_last10 == 1].sum()
    somme_hausse.append(somme)

# Affichage des résultats
print("\nPrécision du modèle sur les 10 derniers jours pour chaque seed :")
for s, acc, somme in zip(seeds, precisions, somme_hausse):
    print(f"Seed {s} : {acc:.2%} | Somme Diff_jour (prédiction Hausse) : {somme:.3f}")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step

Précision du modèle sur les 10 derniers jours pour chaque seed :
Seed 1 : 70.00% | Somme Diff_jour (prédiction Hausse) : 392.490
Seed 444 : 70.00% | Somme Diff_jour (prédiction Hausse) : 603.290
Seed 31 : 80.00% | Somme Diff_jour (prédiction Hausse) : 362.960
Seed 45 : 80.00% | Somme Diff_jour (prédiction Hausse) : 573.760
Seed 5 : 60.00% | Somme Diff_jour (prédiction Hausse) : 307.390


Test sur les random seed 5x

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
import tensorflow as tf
import random
import os


# --- 1. Chargement des données ---
df = pd.read_csv("Données_Diff.csv", parse_dates=True, index_col=0)
df = df.sort_index()

# --- Nettoyage AVANT normalisation ---
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=df.columns)  # supprime toutes les lignes avec NaN/Inf

# --- 2. Préparation des features et target ---
features = df.drop(columns=["result"])
target = df["result"]
y_binary = np.where(target == 1, 1, 0)

# --- 3. Normalisation ---
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(features)

# --- 4. Création des séquences ---
TIME_STEPS = 20
FUTURE_STEPS = 10

def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_binary, TIME_STEPS)
X_train, y_train = X_seq[:-FUTURE_STEPS], y_seq[:-FUTURE_STEPS]

seeds = [1, 2, 3, 4, 5]
nb_repeats = 5

all_precisions = []
all_sommes = []

for repeat in range(nb_repeats):
    precisions = []
    somme_hausse = []
    for seed in seeds:
        np.random.seed(seed + repeat*100)  # pour varier à chaque répétition
        tf.random.set_seed(seed + repeat*100)
        random.seed(seed + repeat*100)
        os.environ['PYTHONHASHSEED'] = str(seed + repeat*100)

        # --- 5. Modèle LSTM ---
        model = Sequential()
        model = Sequential()
        model.add(Input(shape=(TIME_STEPS, X_train.shape[2])))
        model.add(LSTM(64))
        model.add(Dense(1, activation='sigmoid'))
        model.compile(optimizer=Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])
        model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

        # Sélection aléatoire dans les 20% les plus récentes
        nb_seq = len(X_seq)
        start_20p = int(nb_seq * 0.8)
        end_20p = nb_seq - FUTURE_STEPS
        random_start = random.randint(start_20p, end_20p)
        X_test_rand10 = X_seq[random_start:random_start+FUTURE_STEPS]
        y_test_rand10 = y_seq[random_start:random_start+FUTURE_STEPS]

        probas_rand10 = model.predict(X_test_rand10).flatten()
        preds_rand10 = (probas_rand10 > 0.5).astype(int)

        accuracy = np.mean(preds_rand10 == y_test_rand10)
        precisions.append(accuracy)

        idx_rand10 = df.index[random_start+TIME_STEPS:random_start+TIME_STEPS+FUTURE_STEPS]
        diff_jour_rand10 = df.loc[idx_rand10, "Diff_jour"].values
        somme = diff_jour_rand10[preds_rand10 == 1].sum()
        somme_hausse.append(somme)
    all_precisions.extend(precisions)
    all_sommes.extend(somme_hausse)

# Moyennes globales
mean_precision = np.mean(all_precisions)
mean_somme = np.mean(all_sommes)

print(f"\nMoyenne de la précision sur {nb_repeats}x{len(seeds)} runs : {mean_precision:.2%}")
print(f"Moyenne de la somme Diff_jour (Hausse) sur {nb_repeats}x{len(seeds)} runs : {mean_somme:.3f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step

Moyenne de la précision sur 5x5 runs : 52.80%
Moy